In [ ]:
# CELL 1 — DL-POC connection and run name
# Paste ONLY your existing private sf_options = {...} connection block above
# the check below. Obtain it from your working patient-split notebook.
# Credentials are deliberately absent from this file.
# Use the approved compute that successfully ran the patient-split notebook.

if "sf_options" not in globals():
    raise RuntimeError("Add your private sf_options connection setup at the top of this cell.")

sf_options_dl_poc = sf_options.copy()
sf_options_dl_poc.update({
    "sfDatabase": "DSVC_TAKEDA_TA_PRIVATE",
    "sfSchema": "DS_ML",
})
PREFIX = "TAK861_TX_READY_V63_DL_POC"
RUN_ID = "RUN_001"  # Use exactly the same value in training and evaluation.

import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,39}", RUN_ID):
    raise ValueError("RUN_ID must use uppercase letters, numbers and underscores.")
MODEL_TABLE = f"{PREFIX}_MODEL_{RUN_ID}"
EVALUATION_TABLE = f"{PREFIX}_EVAL_{RUN_ID}"
print("DL-POC run:", RUN_ID)


In [ ]:
# CELL 2 — Dependencies and self-contained Snowflake/input helpers
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
try:
    import torch
    import sklearn
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except ModuleNotFoundError as error:
    raise RuntimeError(
        "This notebook needs PyTorch, scikit-learn, NumPy, pandas and matplotlib. "
        "Use an approved ML runtime or install the missing package on your approved compute."
    ) from error
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
"""Embedded by the delivery notebooks; no repository dependency at runtime."""
import base64
import hashlib
import io
import json
import math
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_sf(suffix):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", f"{PREFIX}_{suffix}").load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def checked_metadata(frame, include_split=False):
    columns = ["PATIENT_ID", "END_DT", "RESP"]
    if include_split:
        columns += ["SPLIT", "SPLIT_CONFIG"]
    out = frame.loc[:, columns].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("PATIENT_ID must retain its original nonempty string value.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("END_DT must be a date without an intraday time/timezone.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("RESP must be exactly 0 or 1 before conversion.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys in snapshot metadata.")
    if include_split:
        if set(out.SPLIT) != {"train", "validation", "test"}:
            raise ValueError("Expected the saved train, validation and test assignments.")
        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():
            raise ValueError("Patient overlap between splits.")
        if len(out.SPLIT_CONFIG.unique()) != 1:
            raise ValueError("The split contains inconsistent creation settings.")
        for _, part in out.groupby("SPLIT", observed=True):
            if set(part.RESP) != {0, 1}:
                raise ValueError("Each split must contain both response classes.")
    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)


def validate_metadata_pair(snapshot_frame, manifest_frame, features):
    source = checked_metadata(snapshot_frame)
    manifest = checked_metadata(manifest_frame, include_split=True)
    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Frozen split and source snapshots differ in keys or labels.")
    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])
    # Match the feature-order hash created in the completed split notebook.
    feature_order_hash = hashlib.sha256(
        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()
    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:
        raise ValueError("Feature order or timesteps differ from the frozen split.")
    return manifest, creation


def new_tensor_buffer(metadata, feature_count, seq_len=12):
    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")
    path = Path(temporary.name) / "counts.float32"
    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))
    return temporary, X


def fill_tensor(X, metadata, sequence_rows):
    """Place rows by canonical keys, independent of Spark partition order."""
    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}
    seen = np.zeros(len(metadata), dtype=bool)
    seq_len, feature_count = X.shape[1:]
    for row in sequence_rows:
        raw_date = row["END_DT"]
        if raw_date is None or row["PATIENT_ID"] is None:
            raise ValueError("Null monthly snapshot key.")
        date = pd.Timestamp(raw_date)
        if date.tzinfo is not None or date != date.normalize():
            raise ValueError("Monthly END_DT is not an exact date.")
        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))
        if key not in positions:
            raise ValueError("Unexpected monthly snapshot key.")
        i = positions[key]
        if seen[i] or row["RESP"] != metadata.RESP.iloc[i]:
            raise ValueError("Duplicate monthly snapshot or changed label.")
        sequence = row["SEQUENCE"]
        steps = [month["T"] for month in sequence]
        if len(sequence) != seq_len or any(t is None for t in steps):
            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")
        # Check original values before any integer conversion.
        if sorted(steps) != list(range(seq_len)):
            raise ValueError("Timesteps must be unique integers 0 through 11.")
        sequence = sorted(sequence, key=lambda month: month["T"])
        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)
        if values.shape != (seq_len, feature_count):
            raise ValueError("Monthly feature shape does not match the vocabulary.")
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")
        X[i] = values
        seen[i] = True
    if not seen.all():
        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")
    X.flush()


def input_fingerprints(X, metadata, features):
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    for start in range(0, len(X), 128):
        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {"model_input_sha256": tensor_hash.hexdigest(),
            "snapshot_manifest_sha256": digest_json(records),
            "feature_names_sha256": digest_json(features),
            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}


def load_inputs():
    from pyspark.sql import functions as F
    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid feature names, aliases or order.")
    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()
    frozen = read_sf("PATIENT_SPLIT").select(
        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()
    metadata, creation = validate_metadata_pair(source, frozen, features)
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")
    monthly = read_sf("TENSOR_MONTHLY")
    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
        raise ValueError("Monthly table columns differ from the frozen feature map.")
    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)
    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
        .agg(F.collect_list(F.struct(
            F.col("TIME_STEP").alias("T"),
            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
        )).alias("SEQUENCE"))
        .repartition(128))
    temporary, X = new_tensor_buffer(metadata, len(features))
    try:
        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
        hashes = input_fingerprints(X, metadata, features)
        X.flags.writeable = False
    except BaseException:
        X._mmap.close()
        temporary.cleanup()
        raise
    y = metadata.RESP.to_numpy(dtype=np.float32)
    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)
               for name in ("train", "validation", "test")}
    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)
    return {"X": X, "y": y, "metadata": metadata, "features": features,
            "indices": indices, "hashes": hashes, "split_creation": creation,
            "temporary_directory": temporary}


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")


TRAINING_ARTIFACT_NAMES = {"checkpoint.pt", "training_summary.json", "training_history.csv", "training_history.png"}
if not table_exists(MODEL_TABLE):
    raise FileNotFoundError("Complete training and its save cell for this RUN_ID first.")
if table_exists(EVALUATION_TABLE):
    raise FileExistsError("This run already has a saved TEST report. Reuse the saved report; do not retune using TEST.")
training_artifacts = read_artifacts(MODEL_TABLE, TRAINING_ARTIFACT_NAMES)
training_summary = json.loads(training_artifacts["training_summary.json"])
if training_summary.get("run_id") != RUN_ID or training_summary.get("training_complete") is not True:
    raise ValueError("Saved training run is incomplete or has another RUN_ID.")


In [ ]:
# CELL 3 — Read the frozen DL-POC inputs; validate identity and build raw-count tensor
# This does not calculate TEST predictions or TEST performance.
if "data" in globals():
    old_X = data.get("X")
    if old_X is not None and not old_X._mmap.closed:
        old_X._mmap.close()
        data["temporary_directory"].cleanup()
data = load_inputs()
display(data["metadata"].groupby("SPLIT", observed=True).agg(
    patients=("PATIENT_ID", "nunique"),
    snapshots=("RESP", "size"),
    positive_snapshots=("RESP", "sum"),
))


In [ ]:
# CELL 4 — Restore the completed model and verify the frozen input fingerprints
"""Small sequence classifier; zero-activity months remain real timesteps."""

from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected:
            raise ValueError(f"Expected [batch, {expected[0]}, {expected[1]}] input.")
        if not x.is_floating_point():
            raise TypeError("Transformer input must be floating point.")
        # A zero month is observed absence of activity, not padding. A causal
        # mask is unnecessary because every included month precedes the cutoff.
        hidden = self.projection(x) + self.position
        pooled = self.norm(self.encoder(hidden)).mean(dim=1)
        return self.head(pooled).squeeze(-1)

"""Seed configuration and aggregate-only runtime provenance."""

import os
import platform
import random

import numpy as np
import sklearn
import torch


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Use the DataLoader's seeded generator for each worker process."""
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }

from torch.utils.data import Dataset, DataLoader
class SequenceDataset(Dataset):
    def __init__(self, data, split):
        self.data = data
        self.indices = data["indices"][split]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        x = torch.from_numpy(np.array(self.data["X"][row], dtype=np.float32, copy=True))
        return x, torch.tensor(float(self.data["y"][row]), dtype=torch.float32)


def make_loader(data, split, batch_size, seed, shuffle=False):
    return DataLoader(SequenceDataset(data, split), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))


def checked_log1p(x):
    if not torch.isfinite(x).all() or (x < 0).any():
        raise ValueError("Expected finite nonnegative raw counts before log1p.")
    return torch.log1p(x)


def predict_loader(model, loader, device, criterion=None):
    model.eval()
    labels, scores, total_loss, count = [], [], 0.0, 0
    with torch.inference_mode():
        for x, y in loader:
            x = checked_log1p(x.to(device))
            y = y.to(device)
            logits = model(x)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite validation loss.")
                total_loss += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return total_loss / count, np.concatenate(labels), np.concatenate(scores)


def load_verified_model(blob, data, run_id, device="auto"):
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    if (not isinstance(payload, dict) or payload.get("format_version") != 1
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    if payload.get("input_hashes") != data["hashes"]:
        raise ValueError("Tensor content, keys, labels, feature order or frozen split changed after training.")
    if (payload.get("feature_names") != data["features"]
            or payload.get("time_steps") != list(range(data["X"].shape[1]))
            or payload.get("tensor_shape") != list(data["X"].shape)):
        raise ValueError("Checkpoint tensor layout differs from current inputs.")
    threshold = payload.get("validation_threshold")
    if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    if payload.get("transform") != "log1p":
        raise ValueError("Unknown preprocessing; evaluation stopped.")
    model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved


model, checkpoint, device = load_verified_model(
    training_artifacts["checkpoint.pt"], data, RUN_ID, device="auto")
if (training_summary.get("input_hashes") != checkpoint["input_hashes"]
        or training_summary.get("validation_threshold") != checkpoint["validation_threshold"]):
    raise ValueError("Training summary and checkpoint disagree.")
print("Checkpoint and full input fingerprints verified.")
print("Frozen validation threshold:", checkpoint["validation_threshold"])


In [ ]:
# CELL 5 — Score TEST once using the saved validation threshold
# These are snapshot-level results; repeated patients' snapshots count separately.
# No training, threshold tuning or calibration occurs here.
"""DL-POC aggregate evaluation, suitable for an inline notebook cell.

No repository imports, file writes, threshold tuning, or patient-level exports.
Rates are fractions; lift is relative to the overall evaluated response rate.
"""

import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, precision_recall_curve, roc_curve,
)


def _require(condition, message):
    if not condition:
        raise ValueError(message)


def _real_array(values, name):
    raw = np.asarray(values)
    _require(not np.iscomplexobj(raw) and not (
        raw.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw.flat)
    ), f"{name} must be real, not complex.")
    try:
        return np.asarray(raw, dtype=float)
    except (TypeError, ValueError, OverflowError):
        raise ValueError(f"{name} must contain numeric values.") from None


def _checked_labels_scores(y, scores):
    labels = _real_array(y, "Labels")
    probabilities = _real_array(scores, "Probabilities")
    _require(labels.ndim == probabilities.ndim == 1
             and len(labels) == len(probabilities) and len(labels) > 0,
             "Labels and probabilities must be aligned, nonempty 1-D arrays.")
    _require(np.isfinite(labels).all() and np.isin(np.asarray(y), [0, 1]).all(),
             "Labels must contain only finite binary 0/1 values.")
    _require(np.isfinite(probabilities).all()
             and ((probabilities >= 0) & (probabilities <= 1)).all(),
             "Probabilities must be finite and within [0, 1].")
    return labels.astype(np.int64), probabilities


def classification_metrics(y, scores, threshold):
    """Use a threshold already selected on VALIDATION; never tune on TEST."""
    labels, probabilities = _checked_labels_scores(y, scores)
    cutoff = _real_array(threshold, "Threshold")
    _require(cutoff.ndim == 0 and np.isfinite(cutoff).all()
             and 0 <= float(cutoff) <= 1,
             "Threshold must be a finite scalar in [0, 1].")
    cutoff = float(cutoff)
    predicted = (probabilities >= cutoff).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, probabilities))
        if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, probabilities))
        if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": cutoff,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def _checked_scored_snapshots(frame, score_column):
    _require(isinstance(frame, pd.DataFrame), "Evaluation input must be a pandas DataFrame.")
    _require(isinstance(score_column, str), "Score column must be a column name.")
    _require(frame.columns.is_unique, "Evaluation input contains duplicate column names.")
    keys = ["PATIENT_ID", "END_DT"]
    _require(score_column not in keys + ["RESP"], "Score column must differ from metadata columns.")
    _require(set(keys + ["RESP", score_column]).issubset(frame.columns),
             "Evaluation input is missing snapshot keys, RESP, or scores.")
    out = frame[keys + ["RESP", score_column]].copy()
    _require(not out[keys].isna().any().any(), "Snapshot keys must not be missing.")
    _require(out.PATIENT_ID.map(lambda value: isinstance(value, str)).all(),
             "PATIENT_ID must contain strings to preserve literal identifiers.")
    out["PATIENT_ID"] = out.PATIENT_ID.astype(str)
    _require(out.PATIENT_ID.str.len().gt(0).all()
             and out.PATIENT_ID.str.strip().eq(out.PATIENT_ID).all(),
             "Patient keys must be nonempty with no surrounding whitespace.")
    dates = out.END_DT.astype(str)
    _require(dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(),
             "END_DT must contain ISO calendar dates (YYYY-MM-DD), not intraday timestamps.")
    parsed = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    _require(parsed.notna().all(), "END_DT contains invalid calendar dates.")
    out["END_DT"] = parsed.dt.strftime("%Y-%m-%d")
    _require(not out.duplicated(keys).any(), "Duplicate patient/date snapshots are not allowed.")
    labels, scores = _checked_labels_scores(out.RESP.to_numpy(), out[score_column].to_numpy())
    out["RESP"] = labels
    out[score_column] = scores
    _require(len(out) >= 10, "At least 10 snapshots are required for ten nonempty deciles.")
    return out


def _snapshot_hash(patient, end_date):
    # This public salt gives deterministic, label-independent ties, not anonymity.
    payload = json.dumps(["tak861-targeting-v1", patient, end_date], separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _score_order(frame, score_column):
    # Labels are deliberately inaccessible to ordering logic.
    scoring = frame[["PATIENT_ID", "END_DT", score_column]].rename(
        columns={score_column: "_probability"}
    ).copy()
    scoring["_tie"] = [
        _snapshot_hash(patient, date)
        for patient, date in scoring[["PATIENT_ID", "END_DT"]].itertuples(index=False, name=None)
    ]
    scoring["_position"] = np.arange(len(scoring))
    return scoring.sort_values(
        ["_probability", "_tie", "PATIENT_ID", "END_DT"],
        ascending=[False, True, True, True],
    )["_position"].to_numpy()


def _rank_snapshots(checked, score_column):
    ranked = checked.iloc[_score_order(checked, score_column)].reset_index(drop=True)
    n = len(ranked)
    ranks = np.arange(1, n + 1, dtype=np.int64)
    boundaries = (np.arange(1, 11, dtype=np.int64) * n + 9) // 10
    ranked["decile"] = np.searchsorted(boundaries, ranks, side="left") + 1
    return ranked


def _decile_table(ranked, model_name):
    n, positives = len(ranked), int(ranked.RESP.sum())
    overall_rate = positives / n
    table = ranked.groupby("decile").agg(
        n_snapshots=("RESP", "size"), n_resp1=("RESP", "sum")
    ).reindex(range(1, 11)).reset_index()
    table.insert(0, "model", model_name)
    table["population_fraction"] = table.n_snapshots / n
    table["response_rate"] = table.n_resp1 / table.n_snapshots
    table["decile_precision"] = table.response_rate
    table["share_of_all_resp1"] = table.n_resp1 / positives if positives else np.nan
    table["cumulative_n_snapshots"] = table.n_snapshots.cumsum()
    table["cumulative_population_fraction"] = table.cumulative_n_snapshots / n
    table["cumulative_resp1"] = table.n_resp1.cumsum()
    table["cumulative_recall"] = table.cumulative_resp1 / positives if positives else np.nan
    table["cumulative_precision"] = table.cumulative_resp1 / table.cumulative_n_snapshots
    table["decile_lift"] = table.response_rate / overall_rate if positives else np.nan
    table["cumulative_lift"] = table.cumulative_precision / overall_rate if positives else np.nan
    table["overall_test_response_rate"] = overall_rate
    return table


def _topk_table(deciles):
    selected = deciles.loc[deciles.decile.isin([1, 2, 3])].copy()
    selected["top_k_pct"] = selected.decile * 10
    return selected.rename(columns={
        "cumulative_n_snapshots": "n_selected", "cumulative_resp1": "n_resp1_selected",
        "cumulative_recall": "recall", "cumulative_precision": "precision",
        "cumulative_lift": "lift", "cumulative_population_fraction": "actual_population_fraction",
    })[["model", "top_k_pct", "n_selected", "n_resp1_selected", "recall", "precision",
        "lift", "actual_population_fraction"]].rename(
        columns={"n_resp1_selected": "n_resp1"}
    ).reset_index(drop=True)


def _tie_table(ranked, score_column, model_name):
    scores = ranked[score_column].to_numpy()
    rows = []
    for percentage in (10, 20, 30):
        cutoff = (percentage * len(scores) + 99) // 100
        value = scores[cutoff - 1]
        total = int((scores == value).sum())
        included = int((scores[:cutoff] == value).sum())
        rows.append({
            "model": model_name, "top_k_pct": percentage, "cutoff_score": value,
            "n_tied_at_cutoff": total, "n_tied_selected": included,
            "tie_crosses_boundary": included < total,
        })
    return pd.DataFrame(rows)


def _evaluation_figures(results, checked, score_column):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter

    table, topk = results["deciles"], results["topk"]
    model_name = results["global_metrics"].iloc[0]["model"]
    has_positives = bool(results["global_metrics"].iloc[0]["n_resp1"])
    figures = {}
    color = "#007E87"
    with plt.rc_context({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.facecolor": "white", "axes.facecolor": "white"}):
        fig, ax = plt.subplots(figsize=(8, 5.5), layout="constrained")
        if has_positives:
            ax.plot(np.r_[0, table.cumulative_population_fraction],
                    np.r_[0, table.cumulative_recall], marker="o", markersize=4,
                    linewidth=2.3, color=color, label=model_name)
            ax.plot([0, 1], [0, 1], linestyle="--", color="#777777", label="Random selection")
            ax.legend(loc="lower right")
        else:
            ax.text(.5, .5, "Recall undefined: no positive TEST snapshots", ha="center", transform=ax.transAxes)
        ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="TEST snapshots selected (actual fraction)",
               ylabel="Cumulative RESP=1 captured / recall", title="Cumulative gains | held-out TEST snapshots")
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(alpha=.18)
        figures["gains"] = fig

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
        if has_positives:
            axes[0].bar(table.decile, table.decile_lift, width=.65, color=color)
            axes[1].plot(table.cumulative_population_fraction, table.cumulative_lift,
                         color=color, marker="o", markersize=4, linewidth=2)
        for ax in axes:
            if has_positives:
                ax.axhline(1, color="#777777", linestyle="--", linewidth=1, label="Population baseline = 1")
            else:
                ax.text(.5, .5, "Lift undefined: no positive snapshots", ha="center", transform=ax.transAxes)
            ax.set_ylabel("Lift vs overall TEST response rate")
            ax.grid(axis="y", alpha=.18)
            ax.set_ylim(bottom=0)
        axes[0].set(xticks=range(1, 11), xlabel="Decile (1 = highest propensity)", title="Within-decile lift")
        axes[1].set(xlabel="TEST snapshots selected (actual fraction)", title="Cumulative lift")
        axes[1].xaxis.set_major_formatter(PercentFormatter(1))
        if has_positives:
            axes[1].legend(fontsize=9)
        figures["lift"] = fig

        fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), layout="constrained")
        for ax, metric, title in zip(axes, ["recall", "precision", "lift"],
                                     ["Positive snapshots captured", "Precision in selected snapshots", "Cumulative lift"]):
            values = topk[metric].to_numpy(dtype=float)
            bars = ax.bar(np.arange(3), np.nan_to_num(values, nan=0), width=.60, color=color)
            labels = [(f"{value:.2f}×" if metric == "lift" else f"{value:.1%}")
                      if np.isfinite(value) else "N/A" for value in values]
            ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
            ax.set(xticks=range(3), xticklabels=["Top 10%", "Top 20%", "Top 30%"], title=title)
            ax.set_ylim(bottom=0, top=max(ax.get_ylim()[1] * 1.16, .01))
            if metric != "lift":
                ax.yaxis.set_major_formatter(PercentFormatter(1))
                if ax.get_ylim()[1] > 1:
                    ax.set_yticks(np.arange(0, 1.001, .2))
            ax.grid(axis="y", alpha=.18)
        fig.suptitle(f"{model_name} | held-out TEST targeting performance", fontsize=13)
        figures["topk_performance"] = fig

        fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
        bars = ax.bar(table.decile, table.response_rate, color=color)
        ax.bar_label(bars, labels=[f"{value:.1%}" for value in table.response_rate], padding=3, fontsize=9)
        ax.axhline(float(checked.RESP.mean()), color="#777777", linestyle="--",
                   label="Overall TEST response rate")
        ax.set(xticks=range(1, 11), xlabel="Decile (1 = highest propensity)",
               ylabel="Observed response rate", title=f"{model_name} | response rate by decile",
               ylim=(0, max(table.response_rate.max() * 1.22, .01)))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        if ax.get_ylim()[1] > 1:
            ax.set_yticks(np.arange(0, 1.001, .2))
        ax.legend()
        figures["response_rate"] = fig

        metrics = results["global_metrics"].iloc[0]
        y, scores = checked.RESP.to_numpy(), checked[score_column].to_numpy()
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.9), layout="constrained")
        if has_positives:
            precision, recall, _ = precision_recall_curve(y, scores)
            axes[0].step(recall, precision, where="post", color=color,
                         label=f"Average precision = {metrics['average_precision']:.3f}")
            axes[0].axhline(y.mean(), linestyle="--", color="#777777", label="TEST prevalence")
            axes[0].legend()
        else:
            axes[0].text(.5, .5, "Undefined: no positive TEST snapshots", ha="center", transform=axes[0].transAxes)
        if len(np.unique(y)) == 2:
            fpr, tpr, _ = roc_curve(y, scores)
            axes[1].plot(fpr, tpr, color=color, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
            axes[1].plot([0, 1], [0, 1], linestyle="--", color="#777777")
            axes[1].legend(loc="lower right")
        else:
            axes[1].text(.5, .5, "Undefined: only one TEST class", ha="center", transform=axes[1].transAxes)
        axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision–recall (primary)",
                    xlim=(0, 1), ylim=(0, 1.02))
        axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (supporting)",
                    xlim=(0, 1), ylim=(0, 1.02))
        fig.suptitle("AP is the non-interpolated average precision, not trapezoidal PR-AUC.", fontsize=10)
        figures["discrimination"] = fig

        matrix = np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]], dtype=int)
        fig, ax = plt.subplots(figsize=(5.2, 4.6), layout="constrained")
        ax.imshow(matrix, cmap="Blues", vmin=0)
        for (row, column), value in np.ndenumerate(matrix):
            ax.text(column, row, f"{value:,}", ha="center", va="center", fontsize=18,
                    color="white" if value > matrix.max() / 2 else "#183B4E")
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["RESP=0", "RESP=1"],
               yticklabels=["RESP=0", "RESP=1"], xlabel="Predicted", ylabel="Actual",
               title=f"TEST confusion matrix\nVALIDATION threshold = {metrics['threshold']:.4f}")
        figures["confusion_matrix"] = fig
    return figures


def evaluate_scores(scored_snapshots, threshold, *, score_column="P_RESP1",
                    model_name="Transformer", make_plots=True):
    """Return aggregate frames and figures for an already verified held-out cohort.

    Input: pandas DataFrame with PATIENT_ID (literal string), END_DT (ISO date or
    Python date), RESP, and score_column. threshold must come from VALIDATION.
    Caller must verify exact TEST keys/labels and frozen model provenance before
    this call. Repeated snapshots contribute separately: metrics are snapshot-
    level, not unique-patient capture. Equal scores use a fixed key hash; decile
    boundaries are ceil(d*N/10). No patient rows are returned or persisted.

    Output keys: global_metrics, deciles, topk, ties (pandas DataFrames), and
    figures (gains, lift, topk_performance, response_rate, discrimination,
    confusion_matrix: matplotlib Figure; empty if disabled).
    Caller may display figures and then close them with plt.close(fig).
    """
    checked = _checked_scored_snapshots(scored_snapshots, score_column)
    metrics = classification_metrics(checked.RESP, checked[score_column], threshold)
    ranked = _rank_snapshots(checked, score_column)
    deciles = _decile_table(ranked, model_name)
    base = {
        "model": model_name, "n_test_snapshots": len(checked),
        "n_test_patients": int(checked.PATIENT_ID.nunique()),
        "n_resp1": int(checked.RESP.sum()), "test_response_rate": float(checked.RESP.mean()),
    }
    results = {
        "global_metrics": pd.DataFrame([{**base, **metrics}]),
        "deciles": deciles,
        "topk": _topk_table(deciles),
        "ties": _tie_table(ranked, score_column, model_name),
    }
    results["figures"] = _evaluation_figures(results, checked, score_column) if make_plots else {}
    return results


if data["X"]._mmap.closed:
    raise RuntimeError("Temporary input was released. Reload inputs before scoring.")
test_loader = make_loader(data, "test", checkpoint["training_settings"]["batch_size"],
                          checkpoint["training_settings"]["seed"])
_, test_y, test_scores = predict_loader(model, test_loader, device)
test_rows = data["indices"]["test"]
scored_snapshots = data["metadata"].iloc[test_rows][["PATIENT_ID", "END_DT", "RESP"]].copy()
scored_snapshots = scored_snapshots.reset_index(drop=True)
if not np.array_equal(test_y, scored_snapshots.RESP.to_numpy()):
    raise ValueError("Prediction labels lost alignment with snapshot keys.")
scored_snapshots["P_RESP1"] = test_scores
results = evaluate_scores(scored_snapshots, checkpoint["validation_threshold"], model_name="DL-POC Transformer")
evaluation_metadata = {
    "run_id": RUN_ID,
    "checkpoint_sha256": hashlib.sha256(training_artifacts["checkpoint.pt"]).hexdigest(),
    "input_hashes": data["hashes"],
    "threshold_source": "frozen validation maximum-F1 threshold",
    "threshold": checkpoint["validation_threshold"],
    "evaluation_unit": "patient/date snapshot",
    "test_snapshots": int(len(scored_snapshots)),
    "test_positive_snapshots": int(scored_snapshots.RESP.sum()),
    "ranking_ties": "label-independent snapshot-key hash",
    "clinical_source_review": training_summary.get("source_review"),
    "limitations": "Class-weighted sigmoid scores are not calibrated probabilities; no confidence intervals or LightGBM comparison.",
}
print("TEST inference completed; threshold and model were unchanged.")


In [ ]:
# CELL 6 — Review classification, deciles, top 10/20/30%, ties and charts
# AP is non-interpolated average precision, not trapezoidal PR-AUC.
# Recall uses ALL TEST positives; lift uses TEST prevalence.
# The decile/top-K boundaries use ceil(d * N / 10).
for name in ("global_metrics", "deciles", "topk", "ties"):
    print(name)
    display(results[name])
for name, figure in results["figures"].items():
    display(figure)


In [ ]:
# CELL 7 — Save aggregate TEST report tables and charts to Snowflake
# Patient IDs and per-snapshot predictions are NOT exported by this cell.
evaluation_artifacts = {
    "evaluation_metadata.json": canonical_json(evaluation_metadata).encode("utf-8"),
}
for name in ("global_metrics", "deciles", "topk", "ties"):
    evaluation_artifacts[f"{name}.csv"] = results[name].to_csv(index=False).encode("utf-8")
for name, figure in results["figures"].items():
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    evaluation_artifacts[f"{name}.png"] = buffer.getvalue()
save_artifacts(EVALUATION_TABLE, evaluation_artifacts)


In [ ]:
# CELL 8 — Release temporary input storage after report verification
if read_artifacts(EVALUATION_TABLE, evaluation_artifacts) != evaluation_artifacts:
    raise ValueError("Saved report is not verified; temporary inputs have been retained.")
if not data["X"]._mmap.closed:
    data["X"]._mmap.close()
    data["temporary_directory"].cleanup()
print("DL-POC TEST report saved and verified.")
print("Report table:", f"DSVC_TAKEDA_TA_PRIVATE.DS_ML.{EVALUATION_TABLE}")
